# Download the testing dataset and model parameters

In [ ]:
# please use the following code to download the testing dataset and the model parameters
import gdown
import zipfile
import os



# if you get the following error, please update your gdown by 'pip install --upgrade gdown', and then restart your jupyter
# """
# Access denied with the following error:
#  	Cannot retrieve the public link of the file. You may need to change
# 	the permission to 'Anyone with the link', or have had many accesses. 
# """




# before testing the performance of the proposed model
# you should use the follow anonymous google drive link to download the test dataset


dataset_link = 'https://drive.google.com/uc?id=16SDuZSEoKt4WW76P6yH_Br5lHhXDJ1z1'
output_dataset = 'test.zip'
gdown.download(dataset_link, output_dataset, quiet=False)

parameter_link = 'https://drive.google.com/uc?id=1GUG4v1z-jOtZ3lXlJdmuojrJhfWMjzNh'
output_param = 'params.zip'
gdown.download(parameter_link, output_param, quiet=False)



In [ ]:
with zipfile.ZipFile(output_dataset, 'r') as zip_ref:
    zip_ref.extractall('.')
os.remove(output_dataset)

with zipfile.ZipFile(output_param, 'r') as zip_ref:
    zip_ref.extractall('.')
os.remove(output_param)


# Test Rate-Distortion performance and real encoding / decoding time

In [ ]:
from code.entropy_model_arch import reference_w_checkerboard_n_vpct
from code.g_a_arch import g_a_reference
from code.g_s_arch import g_s_reference
from code.erp2vp_util import VPExtractor
from code.utils import img2tensor
import torch
import numpy as np
import time
from einops import rearrange
import glob
import gdown
import zipfile
import os
import cv2
import math
from torchvision.utils import make_grid
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
g_a = g_a_reference(192, 192)
g_s = g_s_reference(192, 192)

g_a.eval()
g_s.eval()

vpexter = VPExtractor()

In [ ]:
test_image_list = glob.glob('test'+"/*.png")

In [ ]:
def tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):

    if not (torch.is_tensor(tensor) or (isinstance(tensor, list) and all(torch.is_tensor(t) for t in tensor))):
        raise TypeError(f'tensor or list of tensors expected, got {type(tensor)}')

    if torch.is_tensor(tensor):
        tensor = [tensor]
    result = []
    for _tensor in tensor:
        _tensor = _tensor.squeeze(0).float().detach().cpu().clamp_(*min_max)
        _tensor = (_tensor - min_max[0]) / (min_max[1] - min_max[0])

        n_dim = _tensor.dim()
        if n_dim == 4:
            img_np = make_grid(_tensor, nrow=int(math.sqrt(_tensor.size(0))), normalize=False).numpy()
            img_np = img_np.transpose(1, 2, 0)
            if rgb2bgr:
                img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        elif n_dim == 3:
            img_np = _tensor.numpy()
            img_np = img_np.transpose(1, 2, 0)
            if img_np.shape[2] == 1:  # gray image
                img_np = np.squeeze(img_np, axis=2)
            else:
                if rgb2bgr:
                    img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        elif n_dim == 2:
            img_np = _tensor.numpy()
        else:
            raise TypeError(f'Only support 4D, 3D or 2D tensor. But received with dimension: {n_dim}')
        if out_type == np.uint8:
            # Unlike MATLAB, numpy.unit8() WILL NOT round by default.
            img_np = (img_np * 255.0).round()
        img_np = img_np.astype(out_type)
        result.append(img_np)
    if len(result) == 1:
        result = result[0]
    return result


def calculate_psnr(img, img2):
    img = img.astype(np.float64)
    img2 = img2.astype(np.float64)

    mse = np.mean((img - img2)**2)
    if mse == 0:
        return float('inf')
    return 10. * np.log10(255. * 255. / mse)


@torch.no_grad()
def cal_vpsnr(ori_path):
    image_name = os.path.basename(ori_path)

    ori_img = cv2.imread(ori_path)

    ori_vps = vpexter(ori_img)

    x = torch.stack(img2tensor([vp.astype(np.float32) / 255. for vp in ori_vps], bgr2rgb=True, float32=True), 0)

    start = time.time()
    y = g_a(x)
    y = rearrange(y, '(v b) c h w -> v b c h w', b = 1)
    out = entropy.compress(y)
    encode_time = time.time() - start

    start = time.time()
    out = entropy.decompress(out, [len(out), out[0]['y_hat'].size()], cuda=False)
    y_hat = torch.stack([v['y_hat'] for v in out])
    y_hat = rearrange(y_hat, 'v b c h w -> (v b) c h w')
    x_hat = g_s(y_hat)
    decode_time = time.time() - start

    x_numpy = [tensor2img(v) for v in x]
    x_hat_numpy = [tensor2img(v) for v in x_hat]

    vpsnr = np.mean([calculate_psnr(o, r) for o, r in zip(x_numpy, x_hat_numpy)])

    bpi = sum([len(string[0]) for vp_out in out for string in vp_out['strings']]) / 1024.

    return vpsnr, bpi, encode_time, decode_time


In [ ]:
# Warning: Running this cell takes a lot of time, so please be patient

params_root = 'parameters'
param_dirs = list(Path(params_root).iterdir())
param_dirs.sort()

avg_vpsnr_list = []
avg_bpi_list = []
avg_encode_time_list = []
avg_decode_time_list = []


for param_dir in param_dirs:
    # load the parameters
    for param_path in param_dir.iterdir():
        if 'g_a' in str(param_path):
            g_a.load_state_dict(torch.load(str(param_path))['params'])
        elif 'g_s' in str(param_path):
            g_s.load_state_dict(torch.load(str(param_path))['params'])
        elif 'entropy' in str(param_path):
            entropy = reference_w_checkerboard_n_vpct(192, 192)
            entropy.load_state_dict(torch.load(str(param_path))['params'])
            entropy.eval()
            entropy.update()
    
    vpsnr_list = []
    bpi_list = []
    encode_time_list = []
    decode_time_list = []

    for ori_path in test_image_list:
        vpsnr, bpi, encode_time, decode_time = cal_vpsnr(ori_path)
        vpsnr_list.append(vpsnr)
        bpi_list.append(bpi)
        encode_time_list.append(encode_time)
        decode_time_list.append(decode_time)
    
    print("{} : VPSNR: {:.2f}, BPI: {:.2f}, Encode Time: {:.2f}, Decode Time: {:.2f}".format(param_dir.name,np.mean(vpsnr_list), np.mean(bpi_list), np.mean(encode_time_list), np.mean(decode_time_list)))
    
    avg_vpsnr_list.append(np.mean(vpsnr_list))
    avg_bpi_list.append(np.mean(bpi_list))
    avg_encode_time_list.append(np.mean(encode_time_list))
    avg_decode_time_list.append(np.mean(decode_time_list))

# draw the rate-distortion curve 


plt.plot(avg_bpi_list, avg_vpsnr_list, '*-', label = 'Reference+VPCT+Checkerboard')

plt.legend()

plt.title("Test on the VPSNR")
plt.xlabel('BPI (KB)', fontsize=12)
plt.ylabel('VPSNR (dB)', fontsize=12)

plt.grid(True)

plt.show()


# Test Mac per Pixel of VPCT

In [ ]:
from code.vpct_util import VPCTModule
from code.g_a_arch import g_a_reference
from code.g_s_arch import g_s_reference
from thop import profile
import torch
from code.erp2vp_util import VPExtractor
import numpy as np

In [ ]:
g_a = g_a_reference(192, 192)
vpct = VPCTModule(192, 192)
vpexter = VPExtractor()


random_input_numpy = np.random.random((512, 1024, 3))
extracted_vp = vpexter(random_input_numpy)
vp_tensor = torch.from_numpy(np.stack(extracted_vp).transpose(0, 3, 1, 2)).float()


y_vp_tensor = g_a(vp_tensor)
y_vp_tensor = y_vp_tensor.unsqueeze(1)


macs, flops = profile(vpct, inputs=(y_vp_tensor, y_vp_tensor, ))


pixels_num = len(extracted_vp) * extracted_vp[0].shape[0] * extracted_vp[0].shape[1]


RED = "\033[31m"
RESET = "\033[0m"


print(RED + 'The MAC per pixel of the proposed VPCT module is {}K'.format(macs / pixels_num / 1000) + RESET)

# Test input order

In [ ]:
from code.entropy_model_arch import reference_w_checkerboard_n_vpct
from code.g_a_arch import g_a_reference
from code.g_s_arch import g_s_reference
from code.erp2vp_util import VPExtractor
from code.utils import img2tensor
import torch
import numpy as np
import time
from einops import rearrange
import glob
import gdown
import zipfile
import os
import cv2
import math
from torchvision.utils import make_grid
from pathlib import Path
import matplotlib.pyplot as plt
import itertools


In [ ]:
def build_codec(param_dir):
    
    g_a = g_a_reference(192, 192)
    g_s = g_s_reference(192, 192)
    
    for param_path in param_dir.iterdir():
        if 'g_a' in str(param_path):
            g_a.load_state_dict(torch.load(str(param_path))['params'])
        elif 'g_s' in str(param_path):
            g_s.load_state_dict(torch.load(str(param_path))['params'])
        elif 'entropy' in str(param_path):
            entropy = reference_w_checkerboard_n_vpct(192, 192)
            entropy.load_state_dict(torch.load(str(param_path))['params'])
            entropy.eval()
            entropy.update()
    
    g_a.eval()
    g_s.eval()
    entropy.eval()

    return g_a, g_s, entropy

params_root = 'parameters'
param_dirs = list(Path(params_root).iterdir())
param_dirs.sort()

vpexter = VPExtractor()

g_a_0, g_s_0, entropy_0 = build_codec(param_dirs[0])
g_a_1, g_s_1, entropy_1 = build_codec(param_dirs[1])
g_a_2, g_s_2, entropy_2 = build_codec(param_dirs[2])
g_a_3, g_s_3, entropy_3 = build_codec(param_dirs[3])
g_a_4, g_s_4, entropy_4 = build_codec(param_dirs[4])
g_a_5, g_s_5, entropy_5 = build_codec(param_dirs[5])
g_a_6, g_s_6, entropy_6 = build_codec(param_dirs[6])


In [ ]:
test_image_list = glob.glob('test'+"/*.png")

In [ ]:
def tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):

    if not (torch.is_tensor(tensor) or (isinstance(tensor, list) and all(torch.is_tensor(t) for t in tensor))):
        raise TypeError(f'tensor or list of tensors expected, got {type(tensor)}')

    if torch.is_tensor(tensor):
        tensor = [tensor]
    result = []
    for _tensor in tensor:
        _tensor = _tensor.squeeze(0).float().detach().cpu().clamp_(*min_max)
        _tensor = (_tensor - min_max[0]) / (min_max[1] - min_max[0])

        n_dim = _tensor.dim()
        if n_dim == 4:
            img_np = make_grid(_tensor, nrow=int(math.sqrt(_tensor.size(0))), normalize=False).numpy()
            img_np = img_np.transpose(1, 2, 0)
            if rgb2bgr:
                img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        elif n_dim == 3:
            img_np = _tensor.numpy()
            img_np = img_np.transpose(1, 2, 0)
            if img_np.shape[2] == 1:  # gray image
                img_np = np.squeeze(img_np, axis=2)
            else:
                if rgb2bgr:
                    img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        elif n_dim == 2:
            img_np = _tensor.numpy()
        else:
            raise TypeError(f'Only support 4D, 3D or 2D tensor. But received with dimension: {n_dim}')
        if out_type == np.uint8:
            # Unlike MATLAB, numpy.unit8() WILL NOT round by default.
            img_np = (img_np * 255.0).round()
        img_np = img_np.astype(out_type)
        result.append(img_np)
    if len(result) == 1:
        result = result[0]
    return result


def calculate_psnr(img, img2):
    img = img.astype(np.float64)
    img2 = img2.astype(np.float64)

    mse = np.mean((img - img2)**2)
    if mse == 0:
        return float('inf')
    return 10. * np.log10(255. * 255. / mse)

def calculate_mse(img, img2):
    img = img.astype(np.float64)
    img2 = img2.astype(np.float64)

    mse = np.mean((img - img2)**2)
    if mse == 0:
        return float('inf')
    return mse




@torch.no_grad()
def cal_vpsnr(ori_path):
    image_name = os.path.basename(ori_path)

    ori_img = cv2.imread(ori_path)

    ori_vps = vpexter(ori_img)

    x = torch.stack(img2tensor([vp.astype(np.float32) / 255. for vp in ori_vps], bgr2rgb=True, float32=True), 0).cuda()

    start = time.time()
    y = g_a(x)
    y = rearrange(y, '(v b) c h w -> v b c h w', b = 1)
    out = entropy.compress(y)
    encode_time = time.time() - start

    start = time.time()
    out = entropy.decompress(out, [len(out), out[0]['y_hat'].size()])
    y_hat = torch.stack([v['y_hat'] for v in out])
    y_hat = rearrange(y_hat, 'v b c h w -> (v b) c h w')
    x_hat = g_s(y_hat)
    decode_time = time.time() - start

    x_numpy = [tensor2img(v) for v in x]
    x_hat_numpy = [tensor2img(v) for v in x_hat]

    vpsnr = np.mean([calculate_psnr(o, r) for o, r in zip(x_numpy, x_hat_numpy)])

    bpi = sum([len(string[0]) for vp_out in out for string in vp_out['strings']]) / 1024.

    return vpsnr, bpi, encode_time, decode_time

@torch.no_grad()
def cal_vpsnr_order_test_model(ori_vps, g_a, g_s, entropy):
    x = torch.stack(img2tensor([vp.astype(np.float32) / 255. for vp in ori_vps], bgr2rgb=True, float32=True), 0)

    start = time.time()
    y = g_a(x)
    y = rearrange(y, '(v b) c h w -> v b c h w', b = 1)
    out = entropy.compress(y)
    encode_time = time.time() - start

    start = time.time()
    out = entropy.decompress(out, [len(out), out[0]['y_hat'].size()], cuda=False)
    y_hat = torch.stack([v['y_hat'] for v in out])
    y_hat = rearrange(y_hat, 'v b c h w -> (v b) c h w')
    x_hat = g_s(y_hat)
    decode_time = time.time() - start

    x_numpy = [tensor2img(v) for v in x]
    x_hat_numpy = [tensor2img(v) for v in x_hat]

    vpsnr = np.mean([calculate_psnr(o, r) for o, r in zip(x_numpy, x_hat_numpy)])
    vmse = np.mean([calculate_mse(o, r) for o, r in zip(x_numpy, x_hat_numpy)])

    bpi = sum([len(string[0]) for vp_out in out for string in vp_out['strings']]) / 1024.

    return vpsnr, bpi

@torch.no_grad()
def cal_vpsnr_order_test(ori_vps):
    

    x = torch.stack(img2tensor([vp.astype(np.float32) / 255. for vp in ori_vps], bgr2rgb=True, float32=True), 0).cuda()

    start = time.time()
    y = g_a(x)
    y = rearrange(y, '(v b) c h w -> v b c h w', b = 1)
    out = entropy.compress(y)
    encode_time = time.time() - start

    start = time.time()
    out = entropy.decompress(out, [len(out), out[0]['y_hat'].size()])
    y_hat = torch.stack([v['y_hat'] for v in out])
    y_hat = rearrange(y_hat, 'v b c h w -> (v b) c h w')
    x_hat = g_s(y_hat)
    decode_time = time.time() - start

    x_numpy = [tensor2img(v) for v in x]
    x_hat_numpy = [tensor2img(v) for v in x_hat]

    vpsnr = np.mean([calculate_psnr(o, r) for o, r in zip(x_numpy, x_hat_numpy)])
    vmse = np.mean([calculate_mse(o, r) for o, r in zip(x_numpy, x_hat_numpy)])

    bpi = sum([len(string[0]) for vp_out in out for string in vp_out['strings']]) / 1024.

    return vpsnr, vmse, bpi, encode_time, decode_time


In [ ]:
ori_vpsnr_list = []
ori_bpi_list = []
ori_vpsnr_bpi = []

ori_path = test_image_list[0]
image_name = os.path.basename(ori_path)
ori_img = cv2.imread(ori_path)
ori_vps = vpexter(ori_img)
ori_vps_sub_sample = ori_vps[:6]


ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_0, g_s_0, entropy_0))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_1, g_s_1, entropy_1))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_2, g_s_2, entropy_2))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_3, g_s_3, entropy_3))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_4, g_s_4, entropy_4))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_5, g_s_5, entropy_5))
ori_vpsnr_bpi.append(cal_vpsnr_order_test_model(ori_vps_sub_sample, g_a_6, g_s_6, entropy_6))




In [ ]:
# Warning: Running this cell takes a lot of time, so please be patient


from code.bjontegaard_metric import BD_RATE

permutations = itertools.permutations([0, 1, 2, 3, 4, 5])

b_vpsnr_bpi = []
w_vpsnr_bpi = []

ori_vpsnr = np.array(ori_vpsnr_bpi)[:,0]
ori_bpi = np.array(ori_vpsnr_bpi)[:,1]

bbd_rate = 0
wbd_rate = 0

for p in permutations:
    tmp_vpsnr_bpi = []
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_0, g_s_0, entropy_0))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_1, g_s_1, entropy_1))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_2, g_s_2, entropy_2))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_3, g_s_3, entropy_3))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_4, g_s_4, entropy_4))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_5, g_s_5, entropy_5))
    tmp_vpsnr_bpi.append(cal_vpsnr_order_test_model([ori_vps_sub_sample[i] for i in p], g_a_6, g_s_6, entropy_6))

    tmp_vpsnr = np.array(tmp_vpsnr_bpi)[:,0]
    tmp_bpi = np.array(tmp_vpsnr_bpi)[:,1]

    tmp_bd = BD_RATE(tmp_bpi, tmp_vpsnr, ori_bpi, ori_vpsnr)

    if tmp_bd>0:
        if tmp_bd>bbd_rate:
            bbd_rate = tmp_bd
            b_vpsnr_bpi = tmp_vpsnr_bpi
    else:
        if tmp_bd<wbd_rate:
            wbd_rate = tmp_bd
            w_vpsnr_bpi = tmp_vpsnr_bpi



    

In [ ]:
from code.bjontegaard_metric import BD_PSNR

b_vpsnr = np.array(b_vpsnr_bpi)[:, 0]
b_bpi = np.array(b_vpsnr_bpi)[:, 1]
w_vpsnr = np.array(w_vpsnr_bpi)[:, 0]
w_bpi = np.array(w_vpsnr_bpi)[:, 1]

print('Best BD-BR: {}'.format(BD_RATE(ori_bpi, ori_vpsnr, b_bpi, b_vpsnr)))
print('Best BD-VPSNR: {}'.format(BD_PSNR(ori_bpi, ori_vpsnr, b_bpi, b_vpsnr)))

print('Worst BD-BR: {}'.format(BD_RATE(ori_bpi, ori_vpsnr, w_bpi, w_vpsnr)))
print('Worst BD-VPSNR: {}'.format(BD_PSNR(ori_bpi, ori_vpsnr, w_bpi, w_vpsnr)))

